In [ ]:
!pip install supar openai sentencepiece wandb nltk spacy huggingface_hub rouge-score bert-score numpy language_tool_python==2.9.2 scikit-learn scipy tqdm bitsandbytes transformers Pillow ftfy regex flake8 yapf isort yacs gdown tb-nightly future wilds tabulate torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --extra-index-url https://download.pytorch.org/whl/cu124

In [ ]:
import os

# Specify the full path where you want to create the directory
path = "/kaggle/working/logs"

# Create the directory
os.makedirs(path, exist_ok=True)

print(f"Directory created at: {path}")

In [ ]:
#!/usr/bin/env python

import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True' # Add this early

import collections # For deque
# ... other existing imports ...

# Global state for proactive rate limiting for Gemini
GEMINI_API_CALL_TIMESTAMPS = collections.deque()
GEMINI_TOKEN_USAGE_TIMESTAMPS = collections.deque()
GEMINI_MAX_CALLS_PER_MINUTE = 14 # Target 14 RPM (limit is 15 for flash)
GEMINI_MAX_TOKENS_PER_MINUTE = 950000 # Target 950k TPM (limit is 1M for flash)
GEMINI_PROACTIVE_WINDOW_SECONDS = 60.0
GEMINI_DEFAULT_RETRY_DELAY_SECONDS = 30.0
GEMINI_MIN_RETRY_DELAY_SECONDS = 10.0

import time, gc, re, json, random, string, heapq, logging, argparse
import numpy as np
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize # Keep for urdu_sent_tokenize fallback if Stanza fails
# from nltk.tokenize.treebank import TreebankWordDetokenizer # Not strictly needed if urdu_detokenize is custom
from scipy.stats import entropy
import torch # Keep for general torch utilities if any, and for CUDA checks
# from transformers import AutoModelForCausalLM, AutoTokenizer # REMOVE these if only using Gemini for generation
from bert_score import score as bert_score # Keep for evaluation
import difflib
from nltk.metrics.distance import edit_distance
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import matplotlib.pyplot as plt
from huggingface_hub import login # Keep if still used for dataset access or other non-gen models
import unicodedata

# Suppress Warnings
import warnings
warnings.filterwarnings("ignore")
# from transformers import logging as hf_logging # REMOVE if not using transformers.AutoModel etc.
# hf_logging.set_verbosity_error() # REMOVE

from tqdm import tqdm

# Urdu-specific imports
try:
    import stanza
    stanza.download('ur', verbose=False)  # Download Urdu model
except:
    print("Stanza not available, using fallback methods for tokenization/sentencization")

# +++ ADD GEMINI IMPORT +++
import google.generativeai as genai
# +++++++++++++++++++++++++

# Argument Parsing
parser = argparse.ArgumentParser(description='Multi-objective Genetic Search for Prompt Optimization - Urdu')
parser.add_argument('--mode', default="Instruction Only", help='Mode of instructions/prompts')
parser.add_argument('--num-shots', default=0, type=int, help='Number of examples in the prompt')
parser.add_argument('--batch-size', default=1, type=int, help='Batch size (less relevant for Gemini single calls)')
parser.add_argument('--task-idx', default=0, type=int, help='Index of the task')
parser.add_argument('--seed', default=3, type=int, help='Seed for sampling examples')
parser.add_argument('--train-seed', default=42, type=int, help='Seed for splitting and sampling edit operations')
parser.add_argument('--num-compose', default=1, type=int, help='Number of edits composed for one candidate')
parser.add_argument('--num-train', default=250, type=int, help='Number of examples in score set for GA evaluation')
parser.add_argument('--level', default="phrase", help='Level for edit operations')
# parser.add_argument('--simulated-anneal', action='store_true', default=False, help='Use simulated anneal when candidate score <= base score') # Keep if used
parser.add_argument('--agnostic', action='store_true', default=False, help='Use template task-agnostic instruction')
parser.add_argument('--print-orig', action='store_true', default=True, help='Print original instruction and evaluate its performance')
parser.add_argument('--write-preds', action='store_true', default=True, help='Store predictions in a JSON file')
parser.add_argument('--meta-dir', default='/kaggle/working/logs/', help='Directory to store metadata')
parser.add_argument('--meta-name', default='', help='Metadata filename') # Updated name
parser.add_argument('--patience', default=5, type=int, help='Max patience')
parser.add_argument('--num-candidates', default=5, type=int, help='Number of candidates per iteration') # Keep if used
parser.add_argument('--num-iter', default=10, type=int, help='Maximum iterations')
# parser.add_argument('--key-id', default=None, type=int, help='OpenAI key ID if applicable') # REMOVE if not using OpenAI
parser.add_argument('--edits', nargs="+", default=['del', 'swap', 'sub', 'add'], help='Edit operations')
parser.add_argument('--tournament-selection', default=2, type=int, help='Number of tournament selections')
parser.add_argument('--population-size', default=10, type=int, help='Population size for GA')
parser.add_argument('--num-offspring', default=0, type=int, help='Number of offspring per iteration')
parser.add_argument('--mutation-prob', default=0.5, type=float, help='Mutation probability')
parser.add_argument('--data-dir', default='/kaggle/input/urdu-xlsum/', help='Dataset directory')
parser.add_argument('--project-name', default='', help='WandB project name') # Updated name
parser.add_argument('--budget', default=1400, type=int, help='API call budget (Gemini calls can be slower/more expensive)') # Adjusted budget

# +++ ADD GEMINI ARGUMENTS +++
parser.add_argument('--gemini-model-name', default="gemini-2.0-flash", help='Name of the Gemini model to use (e.g., gemini-1.5-pro-latest, gemini-1.5-flash-latest)')
# ++++++++++++++++++++++++++++

args, unknown = parser.parse_known_args()

# Ensure meta_dir exists
os.makedirs(args.meta_dir, exist_ok=True)

# Clear score files at the start of the run
for fname in ['rouge_scores.txt', 'bert_scores.txt', 'bleu_scores.txt']:
    if os.path.exists(os.path.join(args.meta_dir, fname)):
        open(os.path.join(args.meta_dir, fname), 'w').close()
    else:
        with open(os.path.join(args.meta_dir, fname), 'w') as f:
            pass

# +++ CONFIGURE GEMINI API +++
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
gemini_api_key_to_use = user_secrets.get_secret("GEMINI_API_KEY")
if not gemini_api_key_to_use:
    raise ValueError("Gemini API Key must be provided via --gemini-api-key argument or GEMINI_API_KEY environment variable.")
try:
    genai.configure(api_key=gemini_api_key_to_use)
    print(f"Gemini API configured successfully with model: {args.gemini_model_name}")
except Exception as e:
    raise ValueError(f"Failed to configure Gemini API: {e}")

# Initialize Stanza for Urdu
try:
    urdu_nlp = stanza.Pipeline('ur', processors='tokenize,pos,lemma,depparse', verbose=False)
    print("Stanza Urdu pipeline initialized successfully")
except Exception as e:
    urdu_nlp = None
    print(f"Stanza not available or failed to initialize: {e}, using fallback tokenization")

# Initialize progress bar
global_progress_bar = tqdm(total=args.budget, desc="API Calls", leave=False, position=0, dynamic_ncols=True)

try:
    import wandb
    wandb.login(key='') # Replace with your key or use environment variable
    wandb.init(project=args.project_name)
    wandb.config.update(args)
    tqdm.write("Wandb is setup\n")
except Exception as e:
    tqdm.write(f"Wandb initialization error: {e}\n")


evaluation_cache = {}

# Urdu-specific utility functions
def is_urdu_text(text): # Assuming this and enhanced_urdu_tokenize are defined as before
    if not isinstance(text, str): return False
    urdu_range = range(0x0600, 0x06FF)
    return any(ord(char) in urdu_range for char in text)

def enhanced_urdu_tokenize(text): # Assuming this is defined as before
    if not isinstance(text, str) or not text.strip(): return []
    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            return [word.text for sentence in doc.sentences for word in sentence.words if word.text.strip()]
        except Exception: pass
    tokens = re.findall(r'[\u0600-\u06FF\u0750-\u077F]+|[^\s\u0600-\u06FF\u0750-\u077F]+', text)
    return [token.strip() for token in tokens if token.strip()]

def urdu_tokenize(text):
    return enhanced_urdu_tokenize(text)

def urdu_detokenize(tokens): # Assuming this is defined as before
    if not tokens: return ""
    result = tokens[0]
    for token in tokens[1:]:
        if not re.match(r'^[۔؟!،؍]', token) and not unicodedata.category(token[0]).startswith('M'):
            result += " " + token
        else: result += token
    return result

def urdu_sent_tokenize(text): # Assuming this is defined as before
    if not isinstance(text, str): return []
    if urdu_nlp:
        try:
            doc = urdu_nlp(text)
            return [sentence.text for sentence in doc.sentences]
        except Exception: pass
    sentences = re.split(r'([۔؟!])', text)
    result = []
    current_sentence = ""
    for part in sentences:
        current_sentence += part
        if part in '۔؟!':
            result.append(current_sentence.strip()); current_sentence = ""
    if current_sentence.strip(): result.append(current_sentence.strip())
    return [s for s in result if s]


def is_meaningful_phrase(phrase_text: str, is_single_word: bool = False) -> bool:
    if not phrase_text or not phrase_text.strip():
        return False
    
    phrase_cleaned = phrase_text.strip()
    tokens = enhanced_urdu_tokenize(phrase_cleaned)

    if not tokens:
        return False

    if is_single_word:
        if len(tokens) != 1: return False
        if len(phrase_cleaned) < 1: return False
    else:
        if len(tokens) < 2: return False
        if len(phrase_cleaned) < 3: return False

    urdu_char_count = sum(1 for char in phrase_cleaned if '\u0600' <= char <= '\u06FF')
    total_chars_no_space = len(phrase_cleaned.replace(" ", ""))
    if total_chars_no_space == 0: return False
    if urdu_char_count == 0 and total_chars_no_space > 0 : return False
    if urdu_char_count > 0 and urdu_char_count / total_chars_no_space < 0.5: return False

    single_stop_words_ur = {}
    multi_word_stop_phrases_ur = {}

    if is_single_word and phrase_cleaned in single_stop_words_ur:
        return False
    if not is_single_word and phrase_cleaned in multi_word_stop_phrases_ur:
        return False

    if not is_single_word:
        if all(token in single_stop_words_ur for token in tokens):
            return False
            
    digit_count = sum(1 for char in phrase_cleaned if char.isdigit())
    if total_chars_no_space > 0 and (digit_count == total_chars_no_space):
        return False 
    
    is_only_symbols = True
    if total_chars_no_space > 0:
        for char_symbol_check in phrase_cleaned:
            if char_symbol_check.isspace():
                continue
            if char_symbol_check.isalnum() or ('\u0600' <= char_symbol_check <= '\u06FF'):
                is_only_symbols = False
                break
        if is_only_symbols:
            return False
            
    return True

def _gather_all_dependents(
    head_word_obj,
    sentence_words_list,
    consumed_word_ids,
    max_depth=5
):
    phrase_words = {head_word_obj}
    queue = collections.deque([(head_word_obj, 0)])
    visited_word_ids = {head_word_obj.id}

    while queue:
        current_word, depth = queue.popleft()

        if depth >= max_depth:
            continue

        for potential_dependent in sentence_words_list:
            if potential_dependent.head == current_word.id and \
               potential_dependent.id not in visited_word_ids and \
               potential_dependent.id not in consumed_word_ids:
                
                phrase_words.add(potential_dependent)
                visited_word_ids.add(potential_dependent.id)
                queue.append((potential_dependent, depth + 1))

    return phrase_words

def get_constituency_like_phrases_for_urdu(instruction: str) -> list:
    global urdu_nlp
    if not urdu_nlp or not isinstance(instruction, str) or not instruction.strip():
        return []

    final_phrases = []
    try:
        doc = urdu_nlp(instruction)
    except Exception as e:
        tqdm.write(f"Stanza processing error in phrase extraction: {e}. Returning empty list.")
        return []

    for sentence in doc.sentences:
        sentence_words = sentence.words
        consumed_word_ids = set()
        
        for head_word in reversed(sentence_words):
            if head_word.id in consumed_word_ids:
                continue

            if head_word.upos not in ['NOUN', 'PROPN', 'VERB', 'ADJ']:
                continue

            phrase_word_objects = _gather_all_dependents(
                head_word, sentence_words, consumed_word_ids
            )

            if len(phrase_word_objects) > 1:
                sorted_phrase_words = sorted(list(phrase_word_objects), key=lambda w: w.id)
                phrase_text = urdu_detokenize([w.text for w in sorted_phrase_words])

                if is_meaningful_phrase(phrase_text, is_single_word=False):
                    final_phrases.append(phrase_text)
                    for word in phrase_word_objects:
                        consumed_word_ids.add(word.id)

        for word in sentence_words:
            if word.id not in consumed_word_ids:
                if is_meaningful_phrase(word.text, is_single_word=True):
                    final_phrases.append(word.text)

    unique_phrases = sorted(list(set(final_phrases)), key=len, reverse=True)
    
    if not unique_phrases:
        return []

    final_non_overlapping_phrases = []
    unique_phrases.sort(key=lambda p: len(enhanced_urdu_tokenize(p)), reverse=True)

    for phrase in unique_phrases:
        is_sub_phrase = False
        for existing_phrase in final_non_overlapping_phrases:
            if phrase in existing_phrase:
                is_sub_phrase = True
                break
        if not is_sub_phrase:
            final_non_overlapping_phrases.append(phrase)
            
    return final_non_overlapping_phrases

def get_urdu_phrases(instruction):
   return get_constituency_like_phrases_for_urdu(instruction)


def get_urdu_phrase_lookup(base_candidate):
    """Get phrase lookup dictionary for Urdu text"""
    phrases = get_urdu_phrases(base_candidate)
    phrase_lookup = {i: phrase for i, phrase in enumerate(phrases)}
    return phrase_lookup

def normalize_urdu_text(text):
    """Normalize Urdu text for better comparison"""
    if not text: return ""
    text = str(text).strip()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[۔،؍؎؏ؘؙؚؐؑؒؓؔؕؖؗ؛؜]", "", text) # Keep Urdu specific punctuation if needed for ROUGE
    return text.strip()

def calculate_urdu_rouge(reference, generated):
    """Calculate ROUGE scores with proper Urdu text normalization"""
    ref_normalized = normalize_urdu_text(reference)
    gen_normalized = normalize_urdu_text(generated)
    ref_tokens = ref_normalized.split()
    gen_tokens = gen_normalized.split()

    if not ref_tokens or not gen_tokens:
        return {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}

    ref_unigrams = set(ref_tokens)
    gen_unigrams = set(gen_tokens)
    if len(gen_unigrams) == 0 or len(ref_unigrams) == 0:
        rouge1_precision, rouge1_recall, rouge1 = 0.0, 0.0, 0.0
    else:
        rouge1_overlap = len(ref_unigrams.intersection(gen_unigrams))
        rouge1_precision = rouge1_overlap / len(gen_unigrams)
        rouge1_recall = rouge1_overlap / len(ref_unigrams)
        rouge1 = (2 * rouge1_precision * rouge1_recall / (rouge1_precision + rouge1_recall)) if (rouge1_precision + rouge1_recall) > 0 else 0.0

    ref_bigrams = set(zip(ref_tokens, ref_tokens[1:])) if len(ref_tokens) > 1 else set()
    gen_bigrams = set(zip(gen_tokens, gen_tokens[1:])) if len(gen_tokens) > 1 else set()
    if len(gen_bigrams) == 0 or len(ref_bigrams) == 0:
        rouge2_precision, rouge2_recall, rouge2 = 0.0, 0.0, 0.0
    else:
        rouge2_overlap = len(ref_bigrams.intersection(gen_bigrams))
        rouge2_precision = rouge2_overlap / len(gen_bigrams)
        rouge2_recall = rouge2_overlap / len(ref_bigrams)
        rouge2 = (2 * rouge2_precision * rouge2_recall / (rouge2_precision + rouge2_recall)) if (rouge2_precision + rouge2_recall) > 0 else 0.0

    def lcs_length(X, Y):
        m, n = len(X), len(Y)
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        for i in range(1, m + 1):
            for j in range(1, n + 1):
                if X[i - 1] == Y[j - 1]:
                    dp[i][j] = dp[i - 1][j - 1] + 1
                else:
                    dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
        return dp[m][n]

    lcs_len = lcs_length(ref_tokens, gen_tokens)
    if len(gen_tokens) == 0 or len(ref_tokens) == 0:
        rougeL_precision, rougeL_recall, rougeL = 0.0, 0.0, 0.0
    else:
        rougeL_precision = lcs_len / len(gen_tokens)
        rougeL_recall = lcs_len / len(ref_tokens)
        rougeL = (2 * rougeL_precision * rougeL_recall / (rougeL_precision + rougeL_recall)) if (rougeL_precision + rougeL_recall) > 0 else 0.0
    return {"rouge1": rouge1, "rouge2": rouge2, "rougeL": rougeL}

def get_urdu_grammar_score(text):
    """Get grammar score for Urdu text using Gemini."""
    if not is_urdu_text(text) or len(text.strip()) < 5:
        return 0

    # "برائے مہربانی نیچے دیے گئے اردو متن کی گرامر کا جائزہ لیں اور 0 سے 100 کے درمیان ایک عددی اسکور دیں۔ "
        # "صرف نمبر میں جواب دیں۔ کوئی اضافی وضاحت یا متن شامل نہ کریں۔\n\n"
        # "جانچنے کے معیار:\n"
        # "- جملوں کی ساخت اور ترتیب\n"
        # "- الفاظ کا صحیح استعمال\n"
        # "- اعراب اور رموز اوقاف (اگر قابل اطلاق ہوں)\n"
        # "- زبان کی روانی اور مجموعی قواعد\n\n"
        # "متن کا جائزہ لیں:\n"
        # f"'''{text}'''\n\n"
        # "اسکور (صرف نمبر میں):"

    # Prompt for Gemini to score grammar
    prompt_for_grammar = (

        "On a scale of 0 to 100, rate the grammatical correctness of the following Urdu text. "
        "Respond with ONLY the integer score.\n\n"
        f'Urdu Text: "{text}"\n'
        "Score:"
    )

    try:
        # Max output tokens for a score should be very small (e.g., 3-5 for "100")
        # Temperature should be low for deterministic scoring
        raw_output = generate_with_gemini(
            prompt_for_grammar,
            max_output_tokens=10, # Generous for a number
            temperature=0.0,
            operation_name="grammar_scoring_via_gemini"
        )
        numbers = re.findall(r'\b(\d{1,3})\b', raw_output) # Find 1 to 3 digit numbers
        if numbers:
            score = int(numbers[0])
            return max(0, min(100, score))  # Clamp between 0-100
        else:
            # tqdm.write(f"Could not parse score from Gemini grammar output: '{raw_output}' for text: '{text[:50]}...'")
            return urdu_grammar_fallback(text) # Fallback if parsing fails
    except Exception as e:
        tqdm.write(f"Gemini grammar scoring error: {e}")
        return urdu_grammar_fallback(text)

def urdu_grammar_fallback(text):
    """Fallback grammar scoring for Urdu text"""
    score = 100
    if not is_urdu_text(text): score -= 50
    sentences = urdu_sent_tokenize(text)
    if not sentences: score -= 30
    else:
        if len(urdu_tokenize(text)) < 3: score -= 25
    if not any(ending in text for ending in ['۔', '؟', '!']): score -= 20
    for sentence in sentences:
        tokens = urdu_tokenize(sentence)
        if len(tokens) < 2 and len(sentences) > 1 : score -= 10
        elif len(tokens) > 40: score -= 15
    return max(0, score)

def clean_generated_paraphrase(generated_text, original_phrase):
    """Clean and post-process the generated paraphrase"""
    if not generated_text:
        return original_phrase

    # Remove common prefixes/suffixes that LLM might add
    prefixes_to_remove = [
        "متبادل عبارت:", "متبادل:", "جواب:", "answer:",
        "paraphrase:", "alternative:", "یہ ہے:", "یہ:"
    ]

    suffixes_to_remove = ["۔", ".", "!", "؟", "?"]

    cleaned = generated_text.strip()

    # Remove prefixes
    for prefix in prefixes_to_remove:
        if cleaned.lower().startswith(prefix.lower()):
            cleaned = cleaned[len(prefix):].strip()

    # Remove quotes
    cleaned = cleaned.strip('\'"\"\'')

    # Remove suffixes (punctuation)
    for suffix in suffixes_to_remove:
        if cleaned.endswith(suffix):
            cleaned = cleaned[:-len(suffix)].strip()

    # Remove extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned if cleaned else original_phrase

def is_valid_paraphrase(paraphrase, original_phrase, full_context):
    """Validate if the paraphrase is suitable"""
    if not paraphrase or not paraphrase.strip():
        return False

    # Check if it's just the original phrase
    if paraphrase.strip().lower() == original_phrase.strip().lower():
        return False

    # Check if it contains Urdu text
    if not is_urdu_text(paraphrase):
        return False

    # Check length constraints
    original_tokens = urdu_tokenize(original_phrase)
    paraphrase_tokens = urdu_tokenize(paraphrase)

    # Paraphrase shouldn't be too long or too short
    if len(paraphrase_tokens) > len(original_tokens) * 2.5 or len(paraphrase_tokens) < max(1, len(original_tokens) // 2):
        return False

    # Check if paraphrase contains the original phrase (LLM sometimes includes it)
    if original_phrase.lower() in paraphrase.lower() and paraphrase.lower() != original_phrase.lower():
        return False

    # Check if it looks like a complete sentence when it should be a phrase
    if len(original_tokens) <= 3 and any(punct in paraphrase for punct in ['۔', '؟', '!']):
        return False

    return True

def get_gemini_urdu_paraphrase(phrase_to_paraphrase, context_candidate=""):
    """
    Generates a paraphrase for an Urdu phrase using Gemini, optionally with context.
    """
    if not isinstance(phrase_to_paraphrase, str) or not phrase_to_paraphrase.strip():
        return phrase_to_paraphrase # Return original if input is invalid

    # Construct a prompt for Gemini to paraphrase the specific phrase
    # Providing context can help Gemini generate a more suitable paraphrase.
     # "آپ ایک ماہر اردو زبان کے مترادف الفاظ اور جملوں کے ماہر ہیں۔\n"
            # f"نیچے ایک مکمل جملہ دیا گیا ہے: '''{context_candidate}'''\n"
            # f"اس مکمل جملے میں، صرف عبارت '''{phrase_to_paraphrase}''' کا ایک بہتر اور زیادہ موزوں متبادل فراہم کریں۔\n"
            # "آپ کا جواب صرف متبادل عبارت پر مشتمل ہونا چاہیے۔ کوئی اضافی وضاحت، تمہید یا اصل عبارت کا حصہ شامل نہ کریں۔\n"
            # "متبادل عبارت:"
    if context_candidate:
        prompt_for_paraphrase = (
            f'Context: "{context_candidate}"\n'
            f'Phrase to paraphrase: "{phrase_to_paraphrase}"\n\n'
            "Provide a concise Urdu paraphrase for the phrase, considering the context. Respond with only the paraphrased text. Make sure it makes the entire text grammatically correct.\n\n"
            "Paraphrase: "
        )
    else: # Fallback if no context is provided
        prompt_for_paraphrase = (
            f'Phrase: "{phrase_to_paraphrase}"\n\n'
            "Provide a concise Urdu paraphrase for the phrase. Respond with only the paraphrased text.\n\n"
            "Paraphrase: "
        )

    try:
        # Estimate output tokens: allow paraphrase to be a bit longer than the original phrase.
        # For a phrase, 70 tokens should usually be enough.
        # Temperature can be slightly higher for more creative paraphrases.
        paraphrase = generate_with_gemini(
            prompt_for_paraphrase,
            max_output_tokens=70, # Max length of the generated paraphrase
            temperature=0.3,    # Slightly more creative for paraphrasing
            operation_name="paraphrasing_via_gemini"
        )
        return paraphrase.strip() # Gemini output is usually clean, but strip just in case.
    except Exception as e:
        tqdm.write(f"Error during Gemini paraphrasing for '{phrase_to_paraphrase}': {e}")
        return phrase_to_paraphrase # Fallback to original phrase in case of error

def substitute_urdu_phrase(candidate, phrase_to_replace):
    """
    Substitute Urdu phrase with paraphrase using Gemini.
    Returns: (new_candidate_text, actual_paraphrase_used, original_phrase_to_replace)
    """
    if not phrase_to_replace.strip():
        return candidate, phrase_to_replace, phrase_to_replace

    if phrase_to_replace not in candidate:
        # tqdm.write(f"Phrase '{phrase_to_replace}' not found in candidate for substitution.")
        return candidate, phrase_to_replace, phrase_to_replace

    original_phrase_for_return = phrase_to_replace

    try:
        # Get paraphrase using Gemini, providing the full candidate as context
        generated_paraphrase = get_gemini_urdu_paraphrase(phrase_to_replace, context_candidate=candidate)

        # Use your existing cleaning and validation logic
        cleaned_paraphrase = clean_generated_paraphrase(generated_paraphrase, phrase_to_replace)

        if cleaned_paraphrase and is_valid_paraphrase(cleaned_paraphrase, phrase_to_replace, candidate):
            paraphrase_to_use = cleaned_paraphrase

            # Perform substitution
            # Ensure the phrase_to_replace still exists before replacing (it might have been altered by a previous op in a compose chain)
            if phrase_to_replace in candidate:
                temp_candidate = candidate.replace(phrase_to_replace, paraphrase_to_use, 1)
            else: # If original phrase somehow disappeared, don't change candidate
                # tqdm.write(f"Original phrase '{phrase_to_replace}' no longer in candidate during substitution attempt.")
                return candidate, original_phrase_for_return, original_phrase_for_return

            if temp_candidate == candidate: # No effective change (e.g. paraphrase was same as original after cleaning)
                return candidate, original_phrase_for_return, original_phrase_for_return

            grammar_score = get_urdu_grammar_score(temp_candidate) # Use Gemini-based grammar score
            if grammar_score >= 35: # Your grammar threshold (from safe_urdu_mutation)
                return temp_candidate, paraphrase_to_use, original_phrase_for_return
            else:
                # tqdm.write(f"Gemini substitution of '{phrase_to_replace}' with '{paraphrase_to_use}' rejected due to low grammar ({grammar_score}) of new candidate: '{temp_candidate[:50]}...'")
                return candidate, original_phrase_for_return, original_phrase_for_return # Revert if grammar is bad
        else:
            # tqdm.write(f"Gemini paraphrase for '{phrase_to_replace}' ('{generated_paraphrase}') was not valid or no change after cleaning/validation.")
            return candidate, original_phrase_for_return, original_phrase_for_return

    except Exception as e:
        tqdm.write(f"Error during Gemini paraphrasing substitution process for '{phrase_to_replace}': {e}")
        return candidate, original_phrase_for_return, original_phrase_for_return


def delete_urdu_phrase(candidate, phrase):
    """Delete Urdu phrase from candidate"""
    if not phrase.strip(): return candidate
    patterns = [r'\s+' + re.escape(phrase) + r'\s+', r'\s+' + re.escape(phrase), re.escape(phrase) + r'\s+', re.escape(phrase)]
    new_candidate = candidate
    for i, pat_str in enumerate(patterns):
        replacement = ' ' if i == 0 else ''
        temp_candidate, num_subs = re.subn(pat_str, replacement, new_candidate, 1)
        if num_subs > 0:
            new_candidate = temp_candidate
            break
    return ' '.join(new_candidate.split())

def add_urdu_phrase(candidate, phrase_to_add, after_phrase):
    """Add Urdu phrase to candidate after specified phrase or at the beginning"""
    if not phrase_to_add.strip(): return candidate
    if not after_phrase.strip() or after_phrase not in candidate:
        return phrase_to_add + " " + candidate
    else:
        return candidate.replace(after_phrase, after_phrase + " " + phrase_to_add, 1)

def swap_urdu_phrases(candidate, phrase_1, phrase_2):
    """Swap two Urdu phrases in candidate, now with overlap protection."""
    if not phrase_1.strip() or not phrase_2.strip() or phrase_1 == phrase_2:
        return candidate

    idx1 = candidate.find(phrase_1)
    idx2 = candidate.find(phrase_2)

    if idx1 == -1 or idx2 == -1:
        return candidate  # One of the phrases not found

    # --- START: NEW OVERLAP CHECK ---
    end1 = idx1 + len(phrase_1)
    end2 = idx2 + len(phrase_2)

    # Check for any overlap between the two phrase occurrences.
    # This is true if the start of one is before the end of the other.
    if max(idx1, idx2) < min(end1, end2):
        # Phrases overlap, this is unsafe to swap. Return original candidate.
        tqdm.write(
            f"Swap aborted: Phrases overlap. P1='{phrase_1}', P2='{phrase_2}'"
        )
        return candidate
    # --- END: NEW OVERLAP CHECK ---

    # If no overlap, the original logic is safe to proceed.
    placeholder1 = "___PLACEHOLDER_SWAP_1___" + str(random.randint(1000, 9999))
    placeholder2 = "___PLACEHOLDER_SWAP_2___" + str(random.randint(1000, 9999))

    temp_candidate = candidate
    if idx1 < idx2:
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
    else:
        temp_candidate = temp_candidate.replace(phrase_2, placeholder2, 1)
        temp_candidate = temp_candidate.replace(phrase_1, placeholder1, 1)

    final_candidate = temp_candidate.replace(placeholder1, phrase_2, 1)
    final_candidate = final_candidate.replace(placeholder2, phrase_1, 1)

    return final_candidate

def perform_urdu_edit(edit, base_text, phrase_list_full, deleted_phrases_history):
    """Perform edit operation on Urdu text.
    Returns: (mutated_text, new_deleted_phrases_history, edit_description_string)
    """
    mutated = base_text
    phrase_list = list(phrase_list_full)
    edit_description = f"Attempted {edit}: "
    if edit == 'del':
        if not phrase_list:
            edit_description += "No matching Urdu phrase found for deletion."
            return base_text, deleted_phrases_history, edit_description
        chosen = random.choice(phrase_list)
        mutated = delete_urdu_phrase(base_text, chosen)
        if mutated != base_text:
            if chosen not in deleted_phrases_history: deleted_phrases_history.append(chosen)
            edit_description += f"Deleted phrase '{chosen}'."
        else: edit_description += f"Failed to delete phrase '{chosen}'."
    elif edit == 'swap':
        if len(phrase_list) < 2:
            edit_description += "Not enough matching Urdu phrases for swap."
            return base_text, deleted_phrases_history, edit_description
        p1, p2 = random.sample(phrase_list, 2)
        mutated = swap_urdu_phrases(base_text, p1, p2)
        if mutated != base_text: edit_description += f"Swapped '{p1}' with '{p2}'."
        else: edit_description += f"Failed to swap '{p1}' and '{p2}'."
    elif edit == 'sub':
        if not phrase_list:
            edit_description += "No matching Urdu phrase for substitution."
            return base_text, deleted_phrases_history, edit_description
        chosen_phrase_to_replace = random.choice(phrase_list)
        mutated, paraphrase_used, original_phrase = substitute_urdu_phrase(base_text, chosen_phrase_to_replace)
        if mutated != base_text and paraphrase_used != original_phrase:
            edit_description += f"Substituted '{original_phrase}' with '{paraphrase_used}'."
        elif paraphrase_used == original_phrase and mutated != base_text :
             edit_description += f"Replaced '{original_phrase}' with an identical phrase (formatting change)."
        elif paraphrase_used == original_phrase :
            edit_description += f"Substitution of '{original_phrase}' resulted in no change or was reverted."
        else:
            edit_description += f"Substitution of '{original_phrase}' with '{paraphrase_used}' was reverted."
    elif edit == 'add':
        if not deleted_phrases_history:
            edit_description += "No deleted Urdu phrases for addition."
            return base_text, deleted_phrases_history, edit_description
        phrase_to_add = random.choice(deleted_phrases_history)
        after = ""
        if phrase_list:
            after = random.choice(phrase_list)
            edit_description += f"Attempting to add '{phrase_to_add}' after '{after}'."
        else: edit_description += f"Attempting to prepend '{phrase_to_add}'."
        mutated = add_urdu_phrase(base_text, phrase_to_add, after)
        if mutated != base_text:
             if phrase_to_add in deleted_phrases_history: deleted_phrases_history.remove(phrase_to_add)
             edit_description = edit_description.replace("Attempting to add", "Successfully added")
        else: edit_description += " Addition resulted in no change."
    return mutated, deleted_phrases_history, edit_description

def safe_urdu_mutation(edit, base_text, phrase_list, deleted_phrases_history, grammar_threshold=30, meta_file_handle=None, mutation_step_info=""):
    """Safely perform Urdu mutation with grammar checking"""
    mutated, new_del_history, edit_desc = perform_urdu_edit(edit, base_text, phrase_list, deleted_phrases_history)
    full_edit_log = f"    {mutation_step_info} - Op: {edit} - Detail: {edit_desc}"
    if mutated == base_text:
        log_entry = f"{full_edit_log} -> No change to candidate."
        tqdm.write(log_entry)
        if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
        return base_text, deleted_phrases_history
    gscore = get_urdu_grammar_score(mutated)
    if gscore >= grammar_threshold:
        log_entry = f"{full_edit_log} -> Accepted. New Grammar: {gscore}. Candidate: '{mutated[:70]}...'"
    else:
        log_entry = f"{full_edit_log} -> Rejected. Low Grammar: {gscore}. Reverting to: '{base_text[:70]}...'"
        mutated = base_text # Revert
        new_del_history = deleted_phrases_history # Revert history
    tqdm.write(log_entry)
    if meta_file_handle and not meta_file_handle.closed: meta_file_handle.write(log_entry + "\n")
    return mutated, new_del_history

# Instruction Encoding Functions for Urdu
def encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(data_seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))
    random.seed(args.seed)
    test_indices = random.sample(instance_indices, min(number_of_instances, num_available_instances))

    generic_instruction_text = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        # This 'Definition:' part will be part of the user_content for Gemini
        generic_instruction_text = "Definition: " + current_definition.strip()

    promptlist = [] # Will store the list of dicts for get_gemini_summary
    answerlist = []

    
    MAX_ARTICLE_CHARS_FOR_GEMINI = 10000 # Adjust as needed, or remove if truncation isn't desired.

    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_id = instances[i].get('id', f'test_idx_{i}')

        instance_input = raw_instance_input
        if len(raw_instance_input) > MAX_ARTICLE_CHARS_FOR_GEMINI:
            # Using sentence-based truncation for better coherence
            sentences = urdu_sent_tokenize(raw_instance_input) # Assumes urdu_sent_tokenize is available
            truncated_input_parts = []
            current_len = 0
            for sent_text in sentences:
                if not sent_text.strip(): continue
                # Add 1 for space if joining sentences
                if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_FOR_GEMINI:
                    truncated_input_parts.append(sent_text)
                    current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                else:
                    # If the very first sentence is too long, truncate it
                    if not truncated_input_parts and sent_text:
                         truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_FOR_GEMINI])
                    break # Stop adding sentences
            instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_FOR_GEMINI]
            # tqdm.write(f"ID {instance_id}: Truncated input from {len(raw_instance_input)} to {len(instance_input)} chars.")


        # This is the system-level instruction for the summarization task
        system_message_content = ""

        # This is the user-level content, including the specific instruction (definition) and the input text
        user_content_parts = []
        if generic_instruction_text: # This is the evolving prompt part
            user_content_parts.append(generic_instruction_text)
        user_content_parts.append("\nInput:\n" + instance_input) # Clearly label the input
        user_content_parts.append("\nخلاصہ:") # Prompt for the summary
        user_content = "\n".join(user_content_parts)

        # The get_gemini_summary function expects a list of dicts or a single string.
        # We'll pass the list of dicts structure as before.
        prompt_for_gemini_wrapper = [
            {"role": "system", "content": system_message_content},
            {"role": "user", "content": user_content}
        ]
        promptlist.append(prompt_for_gemini_wrapper)

        answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
    return promptlist, answerlist, test_indices

# --- Apply similar truncation logic and prompt construction to `training_encode_urdu_instruction` ---
def training_encode_urdu_instruction(task, instruction_structure=['Definition'], number_of_examples=0, number_of_instances=100, null_word=None, data_seed=0, modified={}, args=args):
    random.seed(args.seed)
    with open(os.path.join(args.data_dir, task), 'r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    instances = data["Instances"]
    num_available_instances = len(instances)
    instance_indices = list(range(num_available_instances))

    actual_num_test_instances = min(number_of_instances, num_available_instances)
    # Ensure test_indices are sampled without replacement and are unique
    if actual_num_test_instances > num_available_instances:
        actual_num_test_instances = num_available_instances # Should not happen if min is used

    test_indices = random.sample(instance_indices, actual_num_test_instances)

    remaining_indices = [i for i in instance_indices if i not in test_indices]

    generic_instruction_text = ''
    current_definition = data.get("Definition", [""])[0]
    if 'Definition' in modified:
        current_definition = modified['Definition']
    if current_definition and current_definition.strip() != '-':
        generic_instruction_text = "Definition: " + current_definition.strip()

    system_message_content = ""
    MAX_ARTICLE_CHARS_FOR_GEMINI = 5000 # Consistent with encode_urdu_instruction

    test_promptlist = []
    test_answerlist = []
    for i in test_indices:
        raw_instance_input = instances[i]['input']
        instance_input = raw_instance_input
        if len(raw_instance_input) > MAX_ARTICLE_CHARS_FOR_GEMINI:
            sentences = urdu_sent_tokenize(raw_instance_input)
            truncated_input_parts = []
            current_len = 0
            for sent_text in sentences:
                if not sent_text.strip(): continue
                if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_FOR_GEMINI:
                    truncated_input_parts.append(sent_text)
                    current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                else:
                    if not truncated_input_parts and sent_text:
                         truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_FOR_GEMINI])
                    break
            instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_FOR_GEMINI]

        user_content_parts = []
        if generic_instruction_text:
            user_content_parts.append(generic_instruction_text)
        user_content_parts.append("\nInput:\n" + instance_input)
        user_content_parts.append("\nخلاصہ:")
        user_content = "\n".join(user_content_parts)

        test_promptlist.append([
            {"role": "system", "content": system_message_content},
            {"role": "user", "content": user_content}
        ])
        test_answerlist.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])

    train_promptlist_full, train_answerlist_full, train_indexlist_full = [], [], []
    random.seed(args.train_seed)
    actual_num_train_samples = min(args.num_train, len(remaining_indices))

    if remaining_indices and actual_num_train_samples > 0 :
        train_sample_indices_from_remaining = random.sample(remaining_indices, actual_num_train_samples)
        for i in train_sample_indices_from_remaining:
            raw_instance_input = instances[i]['input']
            instance_input = raw_instance_input
            if len(raw_instance_input) > MAX_ARTICLE_CHARS_FOR_GEMINI:
                sentences = urdu_sent_tokenize(raw_instance_input)
                truncated_input_parts = []
                current_len = 0
                for sent_text in sentences:
                    if not sent_text.strip(): continue
                    if current_len + len(sent_text) + (1 if truncated_input_parts else 0) <= MAX_ARTICLE_CHARS_FOR_GEMINI:
                        truncated_input_parts.append(sent_text)
                        current_len += len(sent_text) + (1 if len(truncated_input_parts) > 1 else 0)
                    else:
                        if not truncated_input_parts and sent_text:
                             truncated_input_parts.append(sent_text[:MAX_ARTICLE_CHARS_FOR_GEMINI])
                        break
                instance_input = " ".join(truncated_input_parts) if truncated_input_parts else raw_instance_input[:MAX_ARTICLE_CHARS_FOR_GEMINI]

            user_content_parts = []
            if generic_instruction_text:
                user_content_parts.append(generic_instruction_text)
            user_content_parts.append("\nInput:\n" + instance_input)
            user_content_parts.append("\nخلاصہ:")
            user_content = "\n".join(user_content_parts)

            train_promptlist_full.append([
                {"role": "system", "content": system_message_content},
                {"role": "user", "content": user_content}
            ])
            train_answerlist_full.append(instances[i]['output'][0] if isinstance(instances[i]['output'], list) else instances[i]['output'])
            train_indexlist_full.append(i)

    dev_promptlist, dev_answerlist, dev_index_list = [], [], [] # Assuming dev set is not used here
    return test_promptlist, test_answerlist, test_indices, train_promptlist_full, train_answerlist_full, train_indexlist_full, dev_promptlist, dev_answerlist, dev_index_list


def create_batches(instances_list, labels_list=[], batch_size=1):
    instance_batches, label_batches = [], []
    for i in range(0, len(instances_list), batch_size):
        instance_batches.append(instances_list[i:i+batch_size])
        if labels_list: label_batches.append(labels_list[i: i + batch_size])
    return (instance_batches, label_batches) if labels_list else instance_batches

def construct_urdu_instruction_prompt(mode, task_name, num_shots, num_test_instances, data_seed, null_word=None, args=args):
    if mode == "Instruction Only":
        prompt_list, answer_list, index_list = encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word, args=args)
    else: raise ValueError("Invalid mode entry, mode not recognized")
    return prompt_list, answer_list, index_list


def run(mode, batch_size, num_shots, chosen_task_name, num_samples, data_seed=0, override_prompts=False, function_to_call=None, split=None, modified={}, args=args):
    if not override_prompts:
        prompt_list, answer_list, index_list = construct_urdu_instruction_prompt(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, args=args)
    else:
        prompt_list, answer_list, index_list = function_to_call(
            mode=mode, task_name=chosen_task_name, num_shots=num_shots,
            num_test_instances=num_samples, data_seed=data_seed, null_word=None,
            split=split, modified=modified, args=args)

    prompt_batches, batch_test_labels = create_batches(prompt_list, answer_list, batch_size)
    predictions = []

    for batch_of_prompts, batch_of_labels in zip(prompt_batches, batch_test_labels):
        for individual_prompt_messages in batch_of_prompts:
            if generate_with_gemini.count >= args.budget:
                tqdm.write("Budget exhausted in run function. Stopping generation.")
                predictions.extend([""] * (len(answer_list) - len(predictions)))
                break

            pred = get_gemini_summary(individual_prompt_messages)
            predictions.append(pred)

        if generate_with_gemini.count >= args.budget:
            break

    while len(predictions) < len(answer_list):
        predictions.append("")

    all_rouge_scores_dicts = []
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})
            continue
        try: all_rouge_scores_dicts.append(calculate_urdu_rouge(ref, pred_text))
        except Exception as e:
            all_rouge_scores_dicts.append({"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0})

    rouge1_scores_list = [s["rouge1"] for s in all_rouge_scores_dicts]
    rouge2_scores_list = [s["rouge2"] for s in all_rouge_scores_dicts]
    rougeL_scores_list = [s["rougeL"] for s in all_rouge_scores_dicts]

    valid_predictions = [p if p.strip() else "خالی" for p in predictions]
    valid_references = [r if r.strip() else "خالی" for r in answer_list]

    bert_f1_scores_list = [0.0] * len(answer_list)

    if valid_predictions and any(vp.strip() and vp != "خالی" for vp in valid_predictions):
        bert_device = "cuda" if torch.cuda.is_available() else "cpu"
        # Using a robust multilingual model. lang="ur" helps guide layer selection if model supports it.
        # For XLM-R, lang might not be strictly needed but doesn't hurt.
        bert_model_type = "xlm-roberta-large"

        tqdm.write(f"Attempting BERTScore calculation on device: {bert_device} with model: {bert_model_type}")

        try:
            # Attempt with specified device (GPU if available)
            P_bert, R_bert, F1_bert = bert_score(
                valid_predictions,
                valid_references,
                lang="ur", # Specify Urdu
                model_type=bert_model_type,
                device=bert_device,
                verbose=False, # Set to True for more detailed bert-score logs if needed
                batch_size=8 # Adjust batch size based on VRAM / sequence lengths
            )
            bert_f1_scores_list = F1_bert.tolist()
            tqdm.write(f"BERTScore calculation successful on {bert_device}.")
        except Exception as e_gpu:
            tqdm.write(f"Error calculating BERT score on {bert_device} with model {bert_model_type}: {e_gpu}.")
            if bert_device == "cuda":
                tqdm.write("Falling back to CPU for BERTScore calculation.")
                try:
                    P_bert, R_bert, F1_bert = bert_score(
                        valid_predictions,
                        valid_references,
                        lang="ur",
                        model_type=bert_model_type,
                        device="cpu",
                        verbose=False,
                        batch_size=8
                    )
                    bert_f1_scores_list = F1_bert.tolist()
                    tqdm.write("BERTScore calculation successful on CPU (fallback).")
                except Exception as e_cpu:
                    tqdm.write(f"Error calculating BERT score on CPU (fallback) as well: {e_cpu}. Defaulting to 0.")
            else: # Already on CPU and failed
                 tqdm.write(f"BERTScore calculation failed on CPU. Defaulting to 0.")
        # If bert_f1_scores_list is still all zeros after attempts, it means scoring failed.
        if not any(s > 0 for s in bert_f1_scores_list): # Check if any score is non-zero
            tqdm.write("BERTScore resulted in all zeros, indicating a potential issue or all predictions were empty/invalid.")


    bleu_scores_list = []
    smoothie = SmoothingFunction().method4
    for ref, pred_text in zip(answer_list, predictions):
        if not pred_text.strip():
            bleu_scores_list.append(0.0)
            continue
        pred_tokens = urdu_tokenize(pred_text.lower())
        ref_tokens = urdu_tokenize(ref.lower())
        if not pred_tokens or not ref_tokens:
            bleu_scores_list.append(0.0)
            continue
        try:
            bleu_scores_list.append(sentence_bleu([ref_tokens], pred_tokens, smoothing_function=smoothie))
        except Exception as e:
            bleu_scores_list.append(0.0)

    avg_rouge1 = np.mean(rouge1_scores_list) * 100 if rouge1_scores_list else 0.0
    avg_rouge2 = np.mean(rouge2_scores_list) * 100 if rouge2_scores_list else 0.0
    avg_rougeL = np.mean(rougeL_scores_list) * 100 if rougeL_scores_list else 0.0
    avg_bert = np.mean(bert_f1_scores_list) * 100 if bert_f1_scores_list else 0.0
    avg_bleu = np.mean(bleu_scores_list) * 100 if bleu_scores_list else 0.0

    return (predictions, avg_rouge1, avg_rouge2, avg_rougeL, avg_bert, avg_bleu,
            answer_list, index_list, rouge1_scores_list, rouge2_scores_list,
            rougeL_scores_list, bert_f1_scores_list, bleu_scores_list)


# --- MODIFIED/HIGHLIGHTED SECTION ---
def counter(func):
    def wrapper(*args_wrapper, **kwargs_wrapper):
        wrapper.count += 1
        global_progress_bar.update(1) # Assuming global_progress_bar is defined

        # More robust budget check
        if wrapper.count >= args.budget: # args.budget should be defined
            if global_progress_bar.n < global_progress_bar.total:
                # Force progress bar to complete if budget is hit early
                global_progress_bar.n = global_progress_bar.total
                global_progress_bar.refresh()
            if not wrapper.budget_warning_shown: # Show warning only once
                tqdm.write(f"Budget of {args.budget} API calls exhausted (Call #{wrapper.count}). Further API calls may fail or be skipped.")
                wrapper.budget_warning_shown = True
        return func(*args_wrapper, **kwargs_wrapper)
    wrapper.count = 0
    wrapper.budget_warning_shown = False # Initialize the flag
    return wrapper

# Ensure GEMINI_SAFETY_SETTINGS is defined globally as before
GEMINI_SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]

@counter # Apply the counter to this core Gemini call
def generate_with_gemini(prompt_text, max_output_tokens=150, temperature=0.2, operation_name="gemini_generation"):
    """
    Generates text using the configured Gemini model.
    Handles API calls, error handling, proactive and reactive rate limiting (RPM & TPM), and retries.
    Prioritizes API-suggested retry_delay for 429 errors.
    """
    global GEMINI_API_CALL_TIMESTAMPS, GEMINI_TOKEN_USAGE_TIMESTAMPS, args

    # --- Proactive Rate Limiting ---
    # This entire block will run ONCE before the first attempt of a logical call.
    # If it decides to wait, it waits, then proceeds.
    current_time_for_proactive_check = time.time()

    # 1. RPM Check
    while GEMINI_API_CALL_TIMESTAMPS and GEMINI_API_CALL_TIMESTAMPS[0] < current_time_for_proactive_check - GEMINI_PROACTIVE_WINDOW_SECONDS:
        GEMINI_API_CALL_TIMESTAMPS.popleft()

    if len(GEMINI_API_CALL_TIMESTAMPS) >= GEMINI_MAX_CALLS_PER_MINUTE:
        oldest_call_in_window = GEMINI_API_CALL_TIMESTAMPS[0]
        # Calculate how long until the oldest call in the window expires, plus a buffer
        wait_time_rpm = (oldest_call_in_window + GEMINI_PROACTIVE_WINDOW_SECONDS) - current_time_for_proactive_check + 1.0 # 1s buffer
        if wait_time_rpm > 0:
            tqdm.write(f"Proactive RPM limit: Waiting {wait_time_rpm:.2f}s for {operation_name}. Calls in window: {len(GEMINI_API_CALL_TIMESTAMPS)}")
            time.sleep(wait_time_rpm)
        current_time_for_proactive_check = time.time() # Re-evaluate current_time after waiting
        # Re-clean timestamps after waiting
        while GEMINI_API_CALL_TIMESTAMPS and GEMINI_API_CALL_TIMESTAMPS[0] < current_time_for_proactive_check - GEMINI_PROACTIVE_WINDOW_SECONDS:
            GEMINI_API_CALL_TIMESTAMPS.popleft()

    # 2. TPM Check
    while GEMINI_TOKEN_USAGE_TIMESTAMPS and GEMINI_TOKEN_USAGE_TIMESTAMPS[0][0] < current_time_for_proactive_check - GEMINI_PROACTIVE_WINDOW_SECONDS:
        GEMINI_TOKEN_USAGE_TIMESTAMPS.popleft()
    current_tokens_in_window = sum(record[1] for record in GEMINI_TOKEN_USAGE_TIMESTAMPS)

    if current_tokens_in_window >= GEMINI_MAX_TOKENS_PER_MINUTE * 0.90: # If using 90% of token budget
        if GEMINI_TOKEN_USAGE_TIMESTAMPS: # Ensure there are records to base wait time on
            oldest_token_record_ts = GEMINI_TOKEN_USAGE_TIMESTAMPS[0][0]
            wait_time_tpm = (oldest_token_record_ts + GEMINI_PROACTIVE_WINDOW_SECONDS) - current_time_for_proactive_check + 1.5 # 1.5s buffer
            if wait_time_tpm > 0:
                tqdm.write(f"Proactive TPM high ({current_tokens_in_window} tokens): Waiting {wait_time_tpm:.2f}s for {operation_name}.")
                time.sleep(wait_time_tpm)
                # current_time_for_proactive_check = time.time() # Re-evaluate if needed, though RPM check is primary for call initiation
                # Re-clean token timestamps after waiting
                while GEMINI_TOKEN_USAGE_TIMESTAMPS and GEMINI_TOKEN_USAGE_TIMESTAMPS[0][0] < time.time() - GEMINI_PROACTIVE_WINDOW_SECONDS: # Use fresh time.time()
                    GEMINI_TOKEN_USAGE_TIMESTAMPS.popleft()
    # --- End Proactive Rate Limiting ---

    # *** CRITICAL CHANGE: Record the timestamp for this *attempted logical call* for RPM ***
    # This happens AFTER proactive checks pass, but BEFORE the actual API call within the retry loop.
    # This ensures that the proactive RPM limiter knows we are *about to make a call*.
    current_call_attempt_timestamp = time.time()
    GEMINI_API_CALL_TIMESTAMPS.append(current_call_attempt_timestamp)


    model = genai.GenerativeModel(
        args.gemini_model_name,
        safety_settings=GEMINI_SAFETY_SETTINGS
    )
    generation_config = genai.types.GenerationConfig(
        max_output_tokens=max_output_tokens,
        temperature=temperature
    )

    retries = 3
    for attempt in range(retries):
        try:
            # tqdm.write(f"Calling Gemini for {operation_name} (Attempt {attempt + 1}). Prompt: {prompt_text[:100]}...")
            response = model.generate_content(
                prompt_text,
                generation_config=generation_config,
            )

            # Token usage is recorded *after* a successful response, as it comes from metadata
            if response.usage_metadata:
                total_tokens = response.usage_metadata.total_token_count
                if total_tokens > 0:
                    GEMINI_TOKEN_USAGE_TIMESTAMPS.append((time.time(), total_tokens))
            else:
                tqdm.write(f"Warning: Token usage_metadata not found in Gemini response for {operation_name}.")

            if response.parts:
                return response.text.strip()
            elif response.prompt_feedback and response.prompt_feedback.block_reason:
                tqdm.write(f"Gemini API call for {operation_name} blocked. Reason: {response.prompt_feedback.block_reason}")
                # tqdm.write(f"Blocked prompt (first 200 chars): {prompt_text[:200]}...")
                return ""
            else:
                tqdm.write(f"Gemini API call for {operation_name} returned no parts and no block reason. Response: {response}")
                return ""

        except Exception as e:
            tqdm.write(f"Error during Gemini API call for {operation_name} (attempt {attempt + 1}/{retries}): {type(e).__name__} - {e}")

            sleep_duration = -1
            error_str = str(e)

            if "429" in error_str or \
               "rate limit" in error_str.lower() or \
               "resourcehasbeenexhausted" in error_str.lower().replace(" ", "") or \
               "quota" in error_str.lower():

                retry_delay_match = re.search(r"retry_delay\s*{\s*seconds:\s*(\d+)\s*}", error_str, re.IGNORECASE)
                if retry_delay_match:
                    try:
                        api_suggested_delay = int(retry_delay_match.group(1))
                        tqdm.write(f"API suggested retry_delay: {api_suggested_delay}s.")
                        sleep_duration = max(api_suggested_delay, GEMINI_MIN_RETRY_DELAY_SECONDS)
                    except ValueError:
                        tqdm.write(f"Could not parse API retry_delay seconds. Using default: {GEMINI_DEFAULT_RETRY_DELAY_SECONDS}s.")
                        sleep_duration = GEMINI_DEFAULT_RETRY_DELAY_SECONDS
                else:
                    tqdm.write(f"No structured retry_delay in error. Using default for 429/quota: {GEMINI_DEFAULT_RETRY_DELAY_SECONDS}s.")
                    sleep_duration = GEMINI_DEFAULT_RETRY_DELAY_SECONDS

            if attempt < retries - 1:
                if sleep_duration == -1:
                    sleep_duration = (2 ** attempt) + random.uniform(0,1)
                    tqdm.write(f"Retrying {operation_name} in {sleep_duration:.2f} seconds due to other error...")
                else:
                    tqdm.write(f"Retrying {operation_name} in {sleep_duration:.2f} seconds due to rate limit/quota...")

                # Before sleeping for a retry, if this was the first attempt that failed due to rate limit,
                # the RPM timestamp we added might need to be "invalidated" or handled carefully.
                # However, for simplicity, we keep it. The idea is that the *attempt* was made.
                # The subsequent proactive check before the *next logical call* will see this timestamp.
                time.sleep(sleep_duration)
            else:
                tqdm.write(f"Max retries reached for {operation_name}. Last error: {e}")
                # If max retries are reached for a call that was proactively counted,
                # that count remains in GEMINI_API_CALL_TIMESTAMPS. This is correct, as an attempt was made.
                return ""
    return "" # Fallback if all retries fail

def get_gemini_summary(prompt_messages_or_string, max_tokens=None):
    """
    Generates a summary using the configured Gemini model.
    prompt_messages_or_string: Can be a list of chat messages (will be converted) or a single string.
    max_tokens: Specifies the max_output_tokens for the summary.
    """
    prompt_text = ""
    if isinstance(prompt_messages_or_string, list) and all(isinstance(item, dict) for item in prompt_messages_or_string):
        # Convert chat messages to a single string prompt for Gemini
        # Gemini generally prefers a direct instruction style rather than strict chat roles for simple tasks.
        # We can concatenate them, or just use the user prompt if the system prompt is generic.
        system_content = ""
        user_content = ""
        for message in prompt_messages_or_string:
            if message.get("role") == "system":
                system_content += message.get("content", "") + "\n"
            elif message.get("role") == "user":
                user_content += message.get("content", "") + "\n"

        # Construct a single prompt string. You might experiment with formatting.
        # Example: System instructions followed by user input.
        prompt_text = system_content.strip() + "\n\n" + user_content.strip()
        prompt_text = prompt_text.strip() # Ensure no leading/trailing whitespace
    elif isinstance(prompt_messages_or_string, str):
        prompt_text = prompt_messages_or_string
    else:
        tqdm.write(f"Warning: Unexpected prompt format in get_gemini_summary: {type(prompt_messages_or_string)}")
        prompt_text = str(prompt_messages_or_string)

    # Default max_output_tokens for summarization if not specified.
    # This was 50 for Llama. Adjust based on Gemini's typical summary length for your task.
    effective_max_tokens = max_tokens if max_tokens is not None else 75 # Slightly more for Gemini summaries

    return generate_with_gemini(
        prompt_text,
        max_output_tokens=effective_max_tokens,
        temperature=0.3, # Temperature for summarization (0.2-0.5 is usually good)
        operation_name="summarization_via_gemini"
    )


# Initial Setup
meta_path = os.path.join(args.meta_dir, args.meta_name)
# Open meta_file in 'w+' mode at the beginning to truncate/create it.
# It will be opened in 'a' mode within the loop for appending.
meta_file_initial_open = open(meta_path, 'w+', encoding='utf-8')
# We don't write to it immediately here, but it ensures the file is ready.
# It will be closed at the very end of the script.

batch_size = args.batch_size
num_shots = args.num_shots
mode = args.mode
seed = args.seed
train_seed = args.train_seed
model_name = args.gemini_model_name
urdu_task_files = [f for f in os.listdir(args.data_dir) if f.endswith('.json')]
assert args.task_idx >= 0 and args.task_idx < len(urdu_task_files), "Invalid task index"
chosen_task_name = urdu_task_files[args.task_idx]
tqdm.write("Running Experiment for: " + chosen_task_name)
if 'wandb' in globals() and wandb.run:
    wandb.log({"Experiment": f"Running Experiment for: {chosen_task_name} with Gemini-1.5-pro", "Model": model_name})
nltk.download('punkt', quiet=True)
num_samples = 100 # Number of samples for final testing
num_train_samples = args.num_train # Number of samples for GA evaluation
np.random.seed(args.train_seed); torch.manual_seed(args.train_seed)

# add an urdu prompt instruction
instruction = ""

if args.agnostic:
    instruction = "آپ کو ایک متن دیا گیا ہے۔ اسے احتیاط سے پڑھیں اور سمجھیں، اور ایک مختصر خلاصہ فراہم کریں۔"
if 'wandb' in globals() and wandb.run:
    wandb.log({"Num Compose":args.num_compose,"Num Candidates":args.num_candidates,"Max Iterations":args.num_iter,
               "Tournament Selections":args.tournament_selection,"Edit Operations":args.edits,
               "Population Size":args.population_size,"Num Offspring":args.num_offspring,"Patience":args.patience,
               "Mutation Probability":args.mutation_prob})

def evaluate_objectives(candidate_prompt, split="train"):
    cache_key = (candidate_prompt, split)
    if cache_key in evaluation_cache:
        cached_data = evaluation_cache[cache_key]
        # Return the full 7-score tuple
        return (cached_data["summarization_score"], cached_data["grammar_score"], cached_data["avg_rouge1"],
                cached_data["avg_rouge2"], cached_data["avg_rougeL"], cached_data["avg_bert"], cached_data["avg_bleu"])

    if generate_with_gemini.count >= args.budget:
        tqdm.write("Budget exhausted in evaluate_objectives. Returning 0 scores.")
        return (0,) * 7 # Return a tuple of 7 zeros

    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, _, _, _, _, _, _, _) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name,
        num_samples=num_train_samples if split == "train" else num_samples,
        data_seed=args.seed if split == "test" else args.train_seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)

    summarization_score = (0.6 * avg_rL + 0.4 * avg_bert)
    grammar_score = get_urdu_grammar_score(candidate_prompt)

    with open(os.path.join(args.meta_dir, "rouge_scores.txt"), "a", encoding="utf-8") as f_rouge:
        f_rouge.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg R1: {avg_r1:.4f}, Avg R2: {avg_r2:.4f}, Avg RL: {avg_rL:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bert_scores.txt"), "a", encoding="utf-8") as f_bert:
        f_bert.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BERT: {avg_bert:.4f}\n\n")
    with open(os.path.join(args.meta_dir, "bleu_scores.txt"), "a", encoding="utf-8") as f_bleu:
        f_bleu.write(f"Candidate: {candidate_prompt}\nSplit: {split}\nAvg BLEU: {avg_bleu:.4f}\n\n")

    scores = (summarization_score, grammar_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu)
    evaluation_cache[cache_key] = {
        "summarization_score": summarization_score, "grammar_score": grammar_score,
        "avg_rouge1": avg_r1, "avg_rouge2": avg_r2, "avg_rougeL": avg_rL,
        "avg_bert": avg_bert, "avg_bleu": avg_bleu, "split": split}
    return scores

def score_final(candidate_prompt, split="test", write=False):
    (predictions, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu, answer_list, indexlist,
     r1_scores_list, r2_scores_list, rL_scores_list, bert_scores_list, bleu_scores_list) = run(
        mode=args.mode, batch_size=args.batch_size, num_shots=args.num_shots,
        chosen_task_name=chosen_task_name, num_samples=num_samples, data_seed=args.seed,
        override_prompts=True, function_to_call=custom_urdu_instruction_prompt,
        split=split, modified={"Definition": candidate_prompt}, args=args)
    summarization_score = 0.6 * avg_rL + 0.4 * avg_bert
    if split == "test" and write:
        pname = args.meta_name.split(".")[0] + "_predictions.json"
        detailed_predictions_output = []
        for i in range(len(predictions)):
            detailed_predictions_output.append({
                "id": indexlist[i] if i < len(indexlist) else None,
                "reference_summary": answer_list[i] if i < len(answer_list) else None,
                "generated_summary": predictions[i] if i < len(predictions) else None,
                "rouge1": r1_scores_list[i] if i < len(r1_scores_list) else 0.0,
                "rouge2": r2_scores_list[i] if i < len(r2_scores_list) else 0.0,
                "rougeL": rL_scores_list[i] if i < len(rL_scores_list) else 0.0,
                "bert_score": bert_scores_list[i] if i < len(bert_scores_list) else 0.0,
                "bleu_score": bleu_scores_list[i] if i < len(bleu_scores_list) else 0.0,})
        pred_dump = {"final_prompt": candidate_prompt, "overall_avg_rouge1": avg_r1,
                     "overall_avg_rouge2": avg_r2, "overall_avg_rougeL": avg_rL,
                     "overall_avg_bert": avg_bert, "overall_avg_bleu": avg_bleu,
                     "predictions_detailed": detailed_predictions_output}
        ppath = os.path.join(args.meta_dir, pname)
        with open(ppath, "w+", encoding="utf-8") as pfile:
            json.dump(pred_dump, pfile, ensure_ascii=False, indent=2)
    return summarization_score, avg_r1, avg_r2, avg_rL, avg_bert, avg_bleu

def custom_urdu_instruction_prompt(mode=args.mode, task_name=None, num_shots=args.num_shots, num_test_instances=100, data_seed=None, null_word=None, split='train', modified={}, args=args):
    if task_name is None: task_name = chosen_task_name
    if mode == "Instruction Only":
        result = training_encode_urdu_instruction(
            task_name, instruction_structure=["Definition"], number_of_examples=num_shots,
            number_of_instances=num_test_instances, data_seed=data_seed, null_word=null_word,
            modified=modified, args=args)
    else: raise ValueError("Invalid mode for custom_urdu_instruction_prompt")
    if split == 'test': return result[0], result[1], result[2]
    elif split == 'train': return result[3], result[4], result[5]
    else: raise ValueError(f"Invalid split '{split}' for custom_urdu_instruction_prompt")

def tournament_selection(population, population_scores_tuples, num_tournaments):
    parent_indices = random.sample(range(len(population)), k=min(num_tournaments, len(population)))
    if not parent_indices:
        if population: return population[0], population_scores_tuples[0]
        return "", (0.0,) * 7 # Return empty string and a 7-element tuple of zeros

    best_parent_idx = -1
    best_parent_combined_score = -float('inf')
    for idx in parent_indices:
        # The first two scores (summ, gram) are used for selection
        s_score_val = population_scores_tuples[idx][0] if isinstance(population_scores_tuples[idx][0], (int, float)) else 0.0
        g_score_val = population_scores_tuples[idx][1] if isinstance(population_scores_tuples[idx][1], (int, float)) else 0.0
        combined_score = WEIGHT_SUMM * s_score_val + WEIGHT_GRAM * g_score_val
        if combined_score > best_parent_combined_score:
            best_parent_combined_score = combined_score
            best_parent_idx = idx

    if best_parent_idx == -1: # Fallback if all scores were invalid
        best_parent_idx = parent_indices[0]

    return population[best_parent_idx], population_scores_tuples[best_parent_idx]

def urdu_crossover(parent_1_text, parent_2_text):
    try:
        phrases_1 = get_urdu_phrases(parent_1_text)
        phrases_2 = get_urdu_phrases(parent_2_text)
    except Exception: return parent_1_text, True
    if not phrases_1 or not phrases_2: return random.choice([parent_1_text, parent_2_text]), True
    p1_cross_point = random.randint(0, len(phrases_1))
    p2_cross_point = random.randint(0, len(phrases_2))
    offspring_phrases_part1 = phrases_1[:p1_cross_point] + phrases_2[p2_cross_point:]
    offspring_phrases_part2 = phrases_2[:p2_cross_point] + phrases_1[p1_cross_point:]
    chosen_offspring_phrases = random.choice([offspring_phrases_part1, offspring_phrases_part2])
    if not chosen_offspring_phrases: return random.choice([parent_1_text, parent_2_text]), True
    offspring_text = " ".join(chosen_offspring_phrases)
    offspring_text = ' '.join(urdu_tokenize(offspring_text))
    grammar_score = get_urdu_grammar_score(offspring_text)
    if is_urdu_text(offspring_text) and grammar_score >= 30: return offspring_text, False
    else: return random.choice([parent_1_text, parent_2_text]), True

def plot_pareto_front(population_data_tuples, fronts_indices, iteration, meta_dir_path, final=False):
    plt.figure(figsize=(10, 8))
    all_summ_scores = [data[1] for data in population_data_tuples]
    all_gram_scores = [data[2] for data in population_data_tuples]
    plt.scatter(all_summ_scores, all_gram_scores, c='lightgray', alpha=0.6, label='All Population Candidates', s=50)
    colors = ['red', 'blue', 'green', 'purple', 'orange', 'brown', 'pink', 'olive', 'cyan']
    for front_idx, front_candidate_indices in enumerate(fronts_indices):
        if not front_candidate_indices: continue
        color = colors[front_idx % len(colors)]
        front_summ = [population_data_tuples[i][1] for i in front_candidate_indices]
        front_gram = [population_data_tuples[i][2] for i in front_candidate_indices]
        sorted_indices_within_front = np.argsort(front_summ)
        front_summ_sorted = np.array(front_summ)[sorted_indices_within_front]
        front_gram_sorted = np.array(front_gram)[sorted_indices_within_front]
        label = f'Pareto Front {front_idx + 1}' if front_idx > 0 else 'Pareto Optimal Front (F1)'
        plt.scatter(front_summ, front_gram, c=color, label=label, s=70, edgecolors='black')
        if len(front_summ_sorted) > 1 : plt.plot(front_summ_sorted, front_gram_sorted, c=color, linestyle='--', marker='o', markersize=5)
    plt.xlabel('Summarization Score (Higher is Better)'); plt.ylabel('Grammar Score (Higher is Better)')
    title_str = f'Pareto Optimal Fronts (Final)' if final else f'Pareto Optimal Fronts (Iteration {iteration+1})'
    plt.title(title_str, fontsize=16); plt.legend(loc='best'); plt.grid(True, linestyle=':', alpha=0.7); plt.tight_layout()
    plot_name = 'pareto_final.png' if final else f'pareto_iteration_{iteration+1}.png'
    plot_path = os.path.join(meta_dir_path, plot_name)
    plt.savefig(plot_path); plt.close()
    if 'wandb' in globals() and wandb.run:
        try: wandb.log({title_str: wandb.Image(plot_path)}, step=iteration if not final else args.num_iter +1) # Log final plot at a later step
        except Exception as e: tqdm.write(f"WandB logging error for Pareto plot: {e}")
    return plot_path

WEIGHT_SUMM = 0.6
WEIGHT_GRAM = 0.4
BEST_CANDIDATE_WEIGHT_SUMM = 0.8
BEST_CANDIDATE_WEIGHT_GRAM = 0.2

def non_dominated_sorting(population_data_list):
    n = len(population_data_list)
    if n == 0: return []
    domination_count = [0] * n
    dominated_solutions = [[] for _ in range(n)]
    fronts = [[]]
    for i in range(n):
        for j in range(i + 1, n):
            # Indices 1 and 2 are summ and gram scores
            p_summ_raw, p_gram_raw = population_data_list[i][1], population_data_list[i][2]
            q_summ_raw, q_gram_raw = population_data_list[j][1], population_data_list[j][2]

            # Handle non-numeric scores gracefully
            p_summ = p_summ_raw if isinstance(p_summ_raw, (int, float)) else -float('inf')
            p_gram = p_gram_raw if isinstance(p_gram_raw, (int, float)) else -float('inf')
            q_summ = q_summ_raw if isinstance(q_summ_raw, (int, float)) else -float('inf')
            q_gram = q_gram_raw if isinstance(q_gram_raw, (int, float)) else -float('inf')

            p_dominates_q = (p_summ >= q_summ and p_gram >= q_gram) and (p_summ > q_summ or p_gram > q_gram)
            q_dominates_p = (q_summ >= p_summ and q_gram >= p_gram) and (q_summ > p_summ or q_gram > q_gram)
            if p_dominates_q:
                dominated_solutions[i].append(j); domination_count[j] += 1
            elif q_dominates_p:
                dominated_solutions[j].append(i); domination_count[i] += 1
    for i in range(n):
        if domination_count[i] == 0: fronts[0].append(i)
    front_idx = 0
    while fronts[front_idx]:
        next_front_indices = []
        for i in fronts[front_idx]:
            for j in dominated_solutions[i]:
                domination_count[j] -= 1
                if domination_count[j] == 0: next_front_indices.append(j)
        front_idx += 1
        if next_front_indices: fronts.append(next_front_indices)
        else: break
    return fronts

def compute_crowding_distance(population_data_list, front_indices):
    if not front_indices: return {}
    num_in_front = len(front_indices)
    distances = {idx: 0.0 for idx in front_indices}
    num_objectives = 2
    for m in range(num_objectives):
        objective_idx_in_tuple = m + 1 # 1 for summ, 2 for gram
        # Filter out indices with non-numeric scores for this objective
        valid_indices_for_obj_m = [
            i for i in front_indices
            if isinstance(population_data_list[i][objective_idx_in_tuple], (int, float))
        ]
        if not valid_indices_for_obj_m: continue # Skip if no valid scores for this objective

        sorted_front_by_obj_m = sorted(valid_indices_for_obj_m, key=lambda i: population_data_list[i][objective_idx_in_tuple])
        if not sorted_front_by_obj_m: continue

        distances[sorted_front_by_obj_m[0]] = float('inf')
        if len(sorted_front_by_obj_m) > 1: distances[sorted_front_by_obj_m[-1]] = float('inf')

        range_obj_m = 0
        if len(sorted_front_by_obj_m) > 1:
            obj_m_min = population_data_list[sorted_front_by_obj_m[0]][objective_idx_in_tuple]
            obj_m_max = population_data_list[sorted_front_by_obj_m[-1]][objective_idx_in_tuple]
            range_obj_m = obj_m_max - obj_m_min

        if len(sorted_front_by_obj_m) > 2 and range_obj_m > 1e-6:
            for k in range(1, len(sorted_front_by_obj_m) - 1):
                idx_k = sorted_front_by_obj_m[k]
                idx_k_plus_1 = sorted_front_by_obj_m[k+1]
                idx_k_minus_1 = sorted_front_by_obj_m[k-1]
                distances[idx_k] += (population_data_list[idx_k_plus_1][objective_idx_in_tuple] -
                                     population_data_list[idx_k_minus_1][objective_idx_in_tuple]) / range_obj_m
    return distances

def plot_best_candidate_scores(meta_dir_path, r1_scores, r2_scores, rL_scores, b_scores, bl_scores, summ_scores, gram_scores, comb_scores):
    iterations = list(range(len(rL_scores)))
    score_types = {"ROUGE-1": r1_scores, "ROUGE-2": r2_scores, "ROUGE-L": rL_scores,
                   "BERT": b_scores, "BLEU": bl_scores, "Summarization": summ_scores,
                   "Grammar": gram_scores, "Combined (0.9S+0.1G)": comb_scores}
    for score_name, scores_data in score_types.items():
        plt.figure(figsize=(10, 6))
        plt.plot(iterations, scores_data, marker='o', linestyle='-', markersize=5, label=f'Best {score_name}')
        plt.xlabel('Iteration Number (0 = Initial)'); plt.ylabel(f'{score_name} Score')
        plt.title(f'Best Candidate {score_name} Score vs. Iteration', fontsize=14)
        plt.grid(True, linestyle=':', alpha=0.7); plt.xticks(iterations); plt.legend(); plt.tight_layout()
        plot_path = os.path.join(meta_dir_path, f'history_best_{score_name.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", "").replace(".","").replace("+","")}_scores.png')
        plt.savefig(plot_path); plt.close()
        if 'wandb' in globals() and wandb.run:
            try: wandb.log({f"History Best {score_name} Scores Plot": wandb.Image(plot_path)}, step=iterations[-1] if iterations else 0)
            except Exception as e: tqdm.write(f"WandB logging error for score history plot {score_name}: {e}")


def adaptive_mutation_probability(iteration, max_iterations, base_prob=0.5):
    """Adaptive mutation probability that decreases over time"""
    # Start high, decrease as we progress
    decay_factor = 1 - (iteration / max_iterations) * 0.3
    return base_prob * decay_factor

# --- Main Evolutionary Loop ---
W_candidates = [instruction] * args.population_size
W_deletesets = [[] for _ in range(args.population_size)]

with open(meta_path, 'a', encoding='utf-8') as meta_append_initial:
    meta_append_initial.write(f"Initial Urdu Candidate:\t {instruction}\n")
    tqdm.write(f"Initial Urdu Candidate: {instruction}")
    torch.cuda.empty_cache(); gc.collect()

    # Evaluate initial instruction and store all scores
    s_score, g_score, r1_score, r2_score, rL_score, b_score, bl_score = evaluate_objectives(instruction, split='train')
    initial_scores = (s_score, g_score, r1_score, r2_score, rL_score, b_score, bl_score)
    W_full_scores = [initial_scores] * args.population_size

    meta_append_initial.write(
        "Initial Scores (Summ, Gram, R1, R2, RL, BERT, BLEU):\t "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})\n")
    tqdm.write(
        "Initial Scores (Summ, Gram, R1, R2, RL, BERT, BLEU): "
        f"({s_score:.2f}, {g_score:.2f}, {r1_score:.2f}, {r2_score:.2f}, {rL_score:.2f}, {b_score:.2f}, {bl_score:.2f})")

if 'wandb' in globals() and wandb.run:
    wandb.log({"Initial Urdu Candidate": instruction, "Initial Summarization Score": s_score, "Initial Grammar Score": g_score,
               "Initial ROUGE-1 Score": r1_score, "Initial ROUGE-2 Score": r2_score, "Initial ROUGE-L Score": rL_score,
               "Initial BERT Score": b_score, "Initial BLEU Score": bl_score, "Iteration": 0})

history_best_summarization = [s_score]; history_best_grammar = [g_score]
history_best_rouge1 = [r1_score]; history_best_rouge2 = [r2_score]; history_best_rougeL = [rL_score]
history_best_bert = [b_score]; history_best_bleu = [bl_score]
history_best_combined = [BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score]
overall_best_candidate_text = instruction
overall_best_candidate_scores = initial_scores
overall_best_combined_score = BEST_CANDIDATE_WEIGHT_SUMM * s_score + BEST_CANDIDATE_WEIGHT_GRAM * g_score
patience_counter = 0; start_time = time.time(); iter_idx = 0

for iter_idx in range(args.num_iter):
    if generate_with_gemini.count >= args.budget:
        tqdm.write("Budget exhausted before starting iteration. Stopping.")
        break
    with open(meta_path, 'a', encoding='utf-8') as current_iter_meta_file:
        current_iter_meta_file.write(f"\n----- Iteration: {iter_idx + 1} -----\n")
        tqdm.write(f"\n----- Iteration: {iter_idx + 1} -----")
        current_iter_meta_file.write("Population at START of Iteration:\n"); tqdm.write("Population at START of Iteration:")
        population_log_start = []
        for i_pop, cand_text in enumerate(W_candidates):
            s, g, r1, r2, rL, b, bl = W_full_scores[i_pop]
            pop_entry = {"prompt": cand_text, "summ_score": s, "gram_score": g, "r1": r1, "r2": r2, "rL": rL, "bert": b, "bleu": bl}
            population_log_start.append(pop_entry)
            log_line = f"  Cand {i_pop+1}: S={s:.2f}, G={g:.1f}, R1={r1:.2f}, R2={r2:.2f}, RL={rL:.2f}, B={b:.2f}, BL={bl:.2f} - '{cand_text[:40]}...'\n"
            current_iter_meta_file.write(log_line); tqdm.write(log_line.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population Start (Iter {iter_idx+1})": population_log_start}, step=iter_idx+1)

        offspring_candidates, offspring_deletesets = [], []
        if args.num_offspring > 0:
            for j_offspring in range(args.num_offspring // 2):
                if generate_with_gemini.count >= args.budget: break
                parent1_text, _ = tournament_selection(W_candidates, W_full_scores, args.tournament_selection)
                parent2_text, _ = tournament_selection(W_candidates, W_full_scores, args.tournament_selection)
                if parent1_text == parent2_text and len(W_candidates) > 1:
                    temp_population_for_tournament = [c for c in W_candidates if c != parent1_text]
                    if temp_population_for_tournament:
                        temp_objectives_indices = [i for i, c in enumerate(W_candidates) if c != parent1_text]
                        temp_objectives_for_tournament = [W_full_scores[idx] for idx in temp_objectives_indices]
                        if temp_objectives_for_tournament:
                           parent2_text, _ = tournament_selection(temp_population_for_tournament, temp_objectives_for_tournament, args.tournament_selection)
                current_iter_meta_file.write(f"  Parent 1 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent1_text}\n")
                current_iter_meta_file.write(f"  Parent 2 (Iter {iter_idx+1}, OffspringSet {j_offspring+1}):\t {parent2_text}\n")
                child1_text, err1 = urdu_crossover(parent1_text, parent2_text)
                child2_text, err2 = urdu_crossover(parent2_text, parent1_text)
                if not err1 and child1_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child1_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent1_text)]) if parent1_text in W_candidates else [])
                if not err2 and child2_text not in offspring_candidates + W_candidates:
                    offspring_candidates.append(child2_text)
                    offspring_deletesets.append(list(W_deletesets[W_candidates.index(parent2_text)]) if parent2_text in W_candidates else [])

        mutated_candidates_texts, mutated_deletesets = [], []
        candidates_for_mutation = list(W_candidates)
        for i_mut in range(len(candidates_for_mutation)):
            if generate_with_gemini.count >= args.budget: break
            if random.random() < args.mutation_prob:
                base_candidate_text = candidates_for_mutation[i_mut]
                base_deleteset = list(W_deletesets[W_candidates.index(base_candidate_text)])
                current_iter_meta_file.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text}\n")
                tqdm.write(f"  Mutating Candidate {i_mut+1}: {base_candidate_text[:70]}...")
                mutated_text_current = base_candidate_text
                current_deleteset_for_mutation = list(base_deleteset)
                try:
                    phrase_list_for_mutation = get_urdu_phrases(base_candidate_text)
                except Exception as e:
                    current_iter_meta_file.write(f"    Error getting phrases for mutation: {e}\n"); tqdm.write(f"    Error getting phrases: {e}")
                    continue
                for edit_idx_compose in range(args.num_compose):
                    if not args.edits: continue
                    available_edits = list(args.edits)
                    if not phrase_list_for_mutation or len(phrase_list_for_mutation) < 2:
                        if 'swap' in available_edits: available_edits.remove('swap')
                    if not phrase_list_for_mutation:
                        if 'del' in available_edits: available_edits.remove('del')
                        if 'sub' in available_edits: available_edits.remove('sub')
                    if not current_deleteset_for_mutation and 'add' in available_edits: available_edits.remove('add')
                    if not available_edits: break
                    chosen_edit_op = random.choice(available_edits)
                    mutated_text_current, current_deleteset_for_mutation = safe_urdu_mutation(
                        chosen_edit_op, mutated_text_current, list(phrase_list_for_mutation),
                        current_deleteset_for_mutation, meta_file_handle=current_iter_meta_file,
                        mutation_step_info=f"Cand {i_mut+1} ComposeStep {edit_idx_compose+1}")
                    if mutated_text_current != base_candidate_text and edit_idx_compose < args.num_compose - 1 :
                        try: phrase_list_for_mutation = get_urdu_phrases(mutated_text_current)
                        except Exception: phrase_list_for_mutation = []
                if mutated_text_current != base_candidate_text and mutated_text_current not in mutated_candidates_texts + W_candidates + offspring_candidates:
                    mutated_candidates_texts.append(mutated_text_current)
                    mutated_deletesets.append(current_deleteset_for_mutation)

        combined_candidates_texts_pool = W_candidates + offspring_candidates + mutated_candidates_texts
        deletesets_pool_map = {txt: W_deletesets[i] for i, txt in enumerate(W_candidates)}
        deletesets_pool_map.update({txt: offspring_deletesets[i] for i, txt in enumerate(offspring_candidates)})
        deletesets_pool_map.update({txt: mutated_deletesets[i] for i, txt in enumerate(mutated_candidates_texts)})

        unique_combined_texts_list = list(dict.fromkeys(combined_candidates_texts_pool))

        tqdm.write(f"Evaluating {len(unique_combined_texts_list)} unique candidates in combined pool for Iter {iter_idx + 1}")
        all_candidate_data_for_iter = []
        for cand_text_eval in unique_combined_texts_list:
            if generate_with_gemini.count >= args.budget: break
            full_scores_tuple = evaluate_objectives(cand_text_eval, split='train')
            deleteset_val = deletesets_pool_map.get(cand_text_eval, [])
            all_candidate_data_for_iter.append((cand_text_eval, *full_scores_tuple, deleteset_val))

        fronts = non_dominated_sorting(all_candidate_data_for_iter)
        if fronts and fronts[0] and callable(globals().get('plot_pareto_front')):
            plot_pareto_front(all_candidate_data_for_iter, fronts, iter_idx, args.meta_dir)

        next_gen_indices = []
        remaining_population_slots = args.population_size
        for front_indices_in_data in fronts:
            if remaining_population_slots == 0: break
            if len(front_indices_in_data) <= remaining_population_slots:
                next_gen_indices.extend(front_indices_in_data)
                remaining_population_slots -= len(front_indices_in_data)
            else:
                distances = compute_crowding_distance(all_candidate_data_for_iter, front_indices_in_data)
                sorted_front_by_crowding = sorted(front_indices_in_data, key=lambda i: distances.get(i, 0.0), reverse=True)
                next_gen_indices.extend(sorted_front_by_crowding[:remaining_population_slots])
                remaining_population_slots = 0

        temp_W_candidates = [all_candidate_data_for_iter[i][0] for i in next_gen_indices]
        temp_W_full_scores = [all_candidate_data_for_iter[i][1:8] for i in next_gen_indices]
        temp_W_deletesets = [all_candidate_data_for_iter[i][8] for i in next_gen_indices]

        num_actually_selected = len(temp_W_candidates)
        if num_actually_selected < args.population_size:
            padding_needed = args.population_size - num_actually_selected
            if num_actually_selected > 0:
                padding_source = sorted(zip(temp_W_candidates, temp_W_full_scores, temp_W_deletesets), key=lambda x: (WEIGHT_SUMM * x[1][0] + WEIGHT_GRAM * x[1][1]), reverse=True)
                for i_pad in range(padding_needed):
                    source_idx_pad = i_pad % len(padding_source)
                    temp_W_candidates.append(padding_source[source_idx_pad][0])
                    temp_W_full_scores.append(padding_source[source_idx_pad][1])
                    temp_W_deletesets.append(padding_source[source_idx_pad][2])
            else: # Catastrophic case, no candidates survived
                temp_W_candidates.extend([instruction] * padding_needed)
                temp_W_full_scores.extend([initial_scores] * padding_needed)
                temp_W_deletesets.extend([[]] * padding_needed)

        W_candidates = temp_W_candidates
        W_full_scores = temp_W_full_scores
        W_deletesets = temp_W_deletesets

        current_iter_meta_file.write("Population at END of Iteration (Selected for Next Gen):\n"); tqdm.write("Population at END of Iteration (Selected for Next Gen):")
        population_log_end = []
        for i_pop_end, cand_text_end in enumerate(W_candidates):
            s, g, r1, r2, rL, b, bl = W_full_scores[i_pop_end]
            pop_entry_end = {"prompt": cand_text_end, "summ_score": s, "gram_score": g, "r1": r1, "r2": r2, "rL": rL, "bert": b, "bleu": bl}
            population_log_end.append(pop_entry_end)
            log_line_end = f"  Cand {i_pop_end+1}: S={s:.2f}, G={g:.1f}, R1={r1:.2f}, R2={r2:.2f}, RL={rL:.2f}, B={b:.2f}, BL={bl:.2f} - '{cand_text_end[:40]}...'\n"
            current_iter_meta_file.write(log_line_end); tqdm.write(log_line_end.strip())
        if 'wandb' in globals() and wandb.run: wandb.log({f"Population End (Iter {iter_idx+1})": population_log_end}, step=iter_idx+1)

        iter_best_candidate_text, iter_best_scores_tuple = "N/A", (0,)*7
        if fronts and fronts[0]:
            pareto_front_indices = fronts[0]
            best_idx_in_pareto = max(pareto_front_indices, key=lambda i: BEST_CANDIDATE_WEIGHT_SUMM * all_candidate_data_for_iter[i][1] + BEST_CANDIDATE_WEIGHT_GRAM * all_candidate_data_for_iter[i][2])
            iter_best_candidate_text = all_candidate_data_for_iter[best_idx_in_pareto][0]
            iter_best_scores_tuple = all_candidate_data_for_iter[best_idx_in_pareto][1:8]

        history_best_summarization.append(iter_best_scores_tuple[0]); history_best_grammar.append(iter_best_scores_tuple[1])
        history_best_rouge1.append(iter_best_scores_tuple[2]); history_best_rouge2.append(iter_best_scores_tuple[3])
        history_best_rougeL.append(iter_best_scores_tuple[4]); history_best_bert.append(iter_best_scores_tuple[5])
        history_best_bleu.append(iter_best_scores_tuple[6])

        current_iter_combined_score = BEST_CANDIDATE_WEIGHT_SUMM * iter_best_scores_tuple[0] + BEST_CANDIDATE_WEIGHT_GRAM * iter_best_scores_tuple[1]
        history_best_combined.append(current_iter_combined_score)

        tqdm.write(f"Iteration {iter_idx+1} - Best Candidate (0.95S+0.05G): {iter_best_candidate_text[:50]}... | Combined Score: {current_iter_combined_score:.2f}")
        if 'wandb' in globals() and wandb.run:
            wandb.log({"Iteration Best Candidate Text": iter_best_candidate_text,
                       "Iteration Best Summarization Score": iter_best_scores_tuple[0],
                       "Iteration Best Grammar Score": iter_best_scores_tuple[1],
                       "Iteration Best ROUGE-L": iter_best_scores_tuple[4],
                       "Iteration Best BERT": iter_best_scores_tuple[5],
                       "Iteration Best Combined Score": current_iter_combined_score,
                       "API Calls": generate_with_gemini.count, "Iteration": iter_idx + 1})

        if current_iter_combined_score > overall_best_combined_score:
            overall_best_combined_score = current_iter_combined_score
            overall_best_candidate_text = iter_best_candidate_text
            overall_best_candidate_scores = iter_best_scores_tuple
            patience_counter = 0
            tqdm.write(f"New Overall Best Candidate Found! Score: {overall_best_combined_score:.2f}")
        else:
            patience_counter += 1

        if patience_counter >= args.patience:
            tqdm.write(f"Patience ({args.patience}) exhausted. Stopping early.")
            break
    torch.cuda.empty_cache(); gc.collect()
# --- End of Evolutionary Loop ---
with open(meta_path, 'a', encoding='utf-8') as final_meta_file:
    if 'all_candidate_data_for_iter' in locals() and 'fronts' in locals() and fronts and fronts[0]:
         plot_pareto_front(all_candidate_data_for_iter, fronts, iter_idx, args.meta_dir, final=True)

    final_meta_file.write(f"\nSearch Finished.\nAPICalls for search: {generate_with_gemini.count}\n")
    tqdm.write(f"APICalls for search: {generate_with_gemini.count}")
    if 'wandb' in globals() and wandb.run: wandb.log({"Total API Calls (Search)": generate_with_gemini.count})

    plot_best_candidate_scores(args.meta_dir, history_best_rouge1, history_best_rouge2, history_best_rougeL,
                               history_best_bert, history_best_bleu, history_best_summarization,
                               history_best_grammar, history_best_combined)

    tqdm.write("\nTesting the overall best candidate found...")
    final_meta_file.write("\nTesting the overall best candidate found...\n")
    final_s_score, final_r1_score, final_r2_score, final_rL_score, final_b_score, final_bl_score = score_final(
        overall_best_candidate_text, 'test', write=args.write_preds)

    tqdm.write(f"Final Overall Best Candidate (based on 0.95S+0.05G): {overall_best_candidate_text}")
    tqdm.write(f"Final Test Scores (Summarization, R1, R2, RL, BERT, BLEU): ({final_s_score:.2f}, {final_r1_score:.2f}, {final_r2_score:.2f}, {final_rL_score:.2f}, {final_b_score:.2f}, {final_bl_score:.2f})")
    final_meta_file.write(f"Final Overall Best Candidate (based on 0.95S+0.05G): {overall_best_candidate_text}\n")
    final_meta_file.write(f"Final Test Summarization Score: {final_s_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-1 Score: {final_r1_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-2 Score: {final_r2_score:.2f}\n")
    final_meta_file.write(f"Final Test ROUGE-L Score: {final_rL_score:.2f}\n")
    final_meta_file.write(f"Final Test BERT Score: {final_b_score:.2f}\n")
    final_meta_file.write(f"Final Test BLEU Score: {final_bl_score:.2f}\n")

    if 'wandb' in globals() and wandb.run:
        wandb.log({"Final Overall Best Candidate Text": overall_best_candidate_text, "Final Test Summarization Score": final_s_score,
                   "Final Test ROUGE-1 Score": final_r1_score, "Final Test ROUGE-2 Score": final_r2_score,
                   "Final Test ROUGE-L Score": final_rL_score, "Final Test BERT Score": final_b_score,
                   "Final Test BLEU Score": final_bl_score})

    end_time = time.time(); total_time = end_time - start_time
    tqdm.write(f"Total execution time: {total_time:.2f} seconds")
    final_meta_file.write(f"Total execution time: {total_time:.2f} seconds\n")
    if 'wandb' in globals() and wandb.run:
        wandb.log({"Total Execution Time (seconds)": total_time})
        wandb.save(os.path.join(args.meta_dir, args.meta_name))
        if args.write_preds:
            pname = args.meta_name.split('.')[0] + "_predictions.json"
            ppath = os.path.join(args.meta_dir, pname)
            if os.path.exists(ppath): wandb.save(ppath)

global_progress_bar.close()
if 'wandb' in globals() and wandb.run:
    wandb.finish()
if 'meta_file_initial_open' in globals() and meta_file_initial_open and not meta_file_initial_open.closed:
    meta_file_initial_open.close()

torch.cuda.empty_cache()
gc.collect()
tqdm.write("Script finished.")